In [15]:
import os
import shutil
import numpy as np
from obspy import read
from obspy.geodetics import gps2dist_azimuth

# --- 1. 路径与核心参数设置 ---
input_dir = './Wave_Data1'          
select_dir = './P_Select_WaveData'
final_dir = './Final_WaveData'

# 汶川大地震的经纬度 (请根据你的作业指导书确认并微调)
ev_lat = 31.002
ev_lon = 103.367

snr_threshold = 5.0   # SNR 筛选阈值
noise_window = 10.0   # P波到达前 10 秒
signal_window = 5.0   # P波到达后 5 秒

# 创建输出目录
for d in [select_dir, final_dir]:
    if not os.path.exists(d):
        os.makedirs(d)

# --- 2. SNR 计算函数 ---
def calculate_snr(tr, p_arrival):
    try:
        df = tr.stats.sampling_rate
        n_start = int((p_arrival - noise_window - tr.stats.starttime) * df)
        n_end = int((p_arrival - tr.stats.starttime) * df)
        s_start = n_end
        s_end = int((p_arrival + signal_window - tr.stats.starttime) * df)
        
        noise = tr.data[n_start:n_end]
        signal = tr.data[s_start:s_end]
        
        if len(noise) == 0 or len(signal) == 0: return 0
        
        rms_noise = np.sqrt(np.mean(noise**2))
        rms_signal = np.sqrt(np.mean(signal**2))
        
        if rms_noise == 0: return 0
        return rms_signal / rms_noise
    except Exception:
        return 0

# --- 3. 扫描目录并按台站分组 (同时抓取 sac 和 sacpz) ---
all_files = os.listdir(input_dir)
z_sac_files = [f for f in all_files if f.endswith('Z_2008-05-12') or f.endswith('BHZ_2008') or 'BHZ' in f and f.endswith('.sac')]
station_groups = {}

for z_file in z_sac_files:
    file_path = os.path.join(input_dir, z_file)
    st = read(file_path, headonly=True)[0]
    
    net = st.stats.network
    sta = st.stats.station
    group_id = f"{net}.{sta}"
    
    # 提取台站经纬度 (通常 sac 头段里都有 stla 和 stlo)
    sta_lat = st.stats.sac.get('stla')
    sta_lon = st.stats.sac.get('stlo')
    
    # 如果经纬度有效，则计算真实方位角 (az)
    if sta_lat is not None and sta_lon is not None and sta_lat != -12345.0:
        dist_m, az, baz = gps2dist_azimuth(ev_lat, ev_lon, sta_lat, sta_lon)
    else:
        print(f"警告: 台站 {sta} 缺失经纬度，已跳过。")
        continue

    # 读取波形并计算 SNR (固定 P 波在 300s)
    p_arrival_time = 300.0 
    absolute_p_arrival = st.stats.starttime + p_arrival_time
    
    tr = read(file_path)[0]
    tr.filter('bandpass', freqmin=0.5, freqmax=2.0)
    snr = calculate_snr(tr, absolute_p_arrival)
    
    # 找出该台站的所有相关文件 (三个分量的 .sac 和 .sacpz)
    sta_related_files = [f for f in all_files if f.startswith(f"{net}.{sta}.")]
    
    station_groups[group_id] = {
        'files': sta_related_files,
        'snr': snr,
        'az': az,
        'sta_name': sta
    }

# --- 4. SNR 筛选 ---
qualified_groups = []
print(f"开始筛选，共解析了 {len(station_groups)} 个有效台站...")

for gid, info in station_groups.items():
    if info['snr'] >= snr_threshold:
        qualified_groups.append(info)
        # 复制到第一步筛选文件夹
        for f in info['files']:
            shutil.copy(os.path.join(input_dir, f), os.path.join(select_dir, f))

print(f"SNR 筛选完成，保留 {len(qualified_groups)} 个台站 (SNR >= {snr_threshold})。")

# --- 5. 按照真实方位角 (az) 划分 12 个区间 ---
bins = [[] for _ in range(12)]
for group in qualified_groups:
    az = group['az']
    bin_index = int(az % 360 // 30)
    bins[bin_index].append(group)

print("\n--- 开始方位角均衡最优筛选 (基于震中看台站的 Azimuth) ---")
for i, bin_groups in enumerate(bins):
    if not bin_groups:
        print(f"区间 [{i*30:3d}-{(i+1)*30:3d}°]: 无合格数据")
        continue
    
    best_group = max(bin_groups, key=lambda x: x['snr'])
    print(f"区间 [{i*30:3d}-{(i+1)*30:3d}°]: 保留台站 {best_group['sta_name']:>4s} (方位角: {best_group['az']:>6.2f}°, SNR={best_group['snr']:.2f})")
    
    # 将选出台站的所有文件（sac和sacpz）复制到最终文件夹
    for f in best_group['files']:
        shutil.copy(os.path.join(input_dir, f), os.path.join(final_dir, f))

print("\n完美收工！请检查生成的新文件夹。")

开始筛选，共解析了 36 个有效台站...
SNR 筛选完成，保留 30 个台站 (SNR >= 5.0)。

--- 开始方位角均衡最优筛选 (基于震中看台站的 Azimuth) ---
区间 [  0- 30°]: 保留台站 COLA (方位角:  25.44°, SNR=31.13)
区间 [ 30- 60°]: 保留台站 KDAK (方位角:  33.18°, SNR=19.84)
区间 [ 60- 90°]: 保留台站 MIDW (方位角:  69.65°, SNR=9.69)
区间 [ 90-120°]: 保留台站 TARA (方位角:  99.53°, SNR=6.05)
区间 [120-150°]: 保留台站 CTAO (方位角: 135.25°, SNR=37.54)
区间 [150-180°]: 保留台站 MBWA (方位角: 161.07°, SNR=19.41)
区间 [180-210°]: 保留台站 COCO (方位角: 189.32°, SNR=5.47)
区间 [210-240°]: 保留台站 PALK (方位角: 226.65°, SNR=26.84)
区间 [240-270°]: 保留台站 KMBO (方位角: 256.30°, SNR=16.65)
区间 [270-300°]: 无合格数据
区间 [300-330°]: 保留台站 GRFO (方位角: 315.82°, SNR=47.80)
区间 [330-360°]: 保留台站  ALE (方位角: 357.98°, SNR=54.81)

完美收工！请检查生成的新文件夹。
